# 02 Silver tables
I parse each Bronze file into a typed table and check it against itself before anything is compared across files.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200


In [2]:
monthly, monthly_totals = c.read_msitae_monthly()
print(monthly.shape, monthly.month.nunique(), 'months;', monthly.groupby('month').org_code.nunique().min(), 'to', monthly.groupby('month').org_code.nunique().max(), 'providers per month')
# Internal check: provider rows must sum to the TOTAL row that each file carries. I expect zero everywhere.
cols = c.ATT_COLS + c.BOOKED_COLS + c.OVER4_COLS + c.OVER4_BOOKED_COLS
chk = monthly.groupby('month')[cols].sum() - monthly_totals.set_index('month')[cols]
print('max absolute difference between provider rows and file TOTAL row:', int(chk.abs().max().max()))
assert chk.abs().max().max() == 0

(2386, 17) 12 months; 197 to 201 providers per month
max absolute difference between provider rows and file TOTAL row: 0


In [3]:
cqi = c.read_cqi()
print(cqi.shape, cqi.month.nunique(), 'months', cqi.org_code.nunique(), 'orgs', cqi.suppressed.sum(), 'suppressed rows')
# Internal check: CQI provider rows should sum to its own ENG row. I look at the MSitAE and ECDS counts only.
for m in ['AEQI011', 'AEQI012']:
    s = cqi[cqi.measure_id == m]
    print(m, 'ENG', s[s.org_code == 'ENG'].measure_value.sum(), 'providers', s[s.org_code != 'ENG'].measure_value.sum())

(46266, 7) 12 months 186 orgs 894 suppressed rows
AEQI011 ENG 26631296.0 providers 26631300.0
AEQI012 ENG 27139660.0 providers 27139660.0


In [4]:
t1, t3, t6, t11, op1 = c.read_msitae_table1(), c.read_msitae_table3(), c.read_msitae_table6(), c.read_msitae_table11(), c.read_op_summary1()
print(len(t1), 'years in Table 1;', len(t3), 'in Table 3;', len(t6), 'providers in Table 6;', len(op1), 'years in outpatient Summary Report 1')
t11

15 years in Table 1; 15 in Table 3; 121 providers in Table 6; 11 years in outpatient Summary Report 1


,year,dept_type,ecds,msitae,ecds_pct_of_msitae
0,2025-26,Type 1 - Consultant led A&E,17287155,17121495,1.0097
1,2025-26,Type 2 - Single specialty A&E,504892,611420,0.8258
2,2025-26,Types 3 - Minor Injury/Walk in units,8847261,10243110,0.8637
3,2025-26,Total (Excluding planned follow-up)¹,26639308,27976025,0.9522


In [5]:
out = {'msitae_monthly_provider': monthly, 'msitae_monthly_totals': monthly_totals, 'cqi_long': cqi,
       'msitae_table1_attendances': t1, 'msitae_table3_four_hour': t3, 'msitae_table6_type1_provider': t6,
       'msitae_table11_dept_type': t11, 'op_summary1_appointments': op1}
for name, df in out.items():
    df.to_parquet(c.SILVER / f'{name}.parquet', index=False)
    print(f'{name:34s}{len(df):>8,d} rows')

msitae_monthly_provider              2,386 rows
msitae_monthly_totals                   12 rows
cqi_long                            46,266 rows
msitae_table1_attendances               15 rows
msitae_table3_four_hour                 15 rows
msitae_table6_type1_provider           121 rows
msitae_table11_dept_type                 4 rows
op_summary1_appointments                11 rows
